In [196]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.nn import functional as F

In [197]:
batch_size = 32 #indep sequence processing in parallel
block_size = 8 #maximum context length for prediction
max_iters = 3000
eval_interval = 300
lr = 1e-2
device ='cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

In [198]:
print(device)

cuda


In [199]:
torch.manual_seed(3407)

In [200]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Length (char):", len(text))
print(text[:500])

Length (char): 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [201]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [202]:
print(vocab_size)

65


In [203]:
str_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_str = { i:ch for i, ch in enumerate(chars) }

In [204]:
encode = lambda s: [str_to_int[c] for c in s] #take a str and output a list of int

In [205]:
decode = lambda l: ''.join([int_to_str[i] for i in l]) #vice versa

In [206]:
print(encode("Meoww"))

[25, 43, 53, 61, 61]


In [207]:
print(decode([25, 43, 53, 61, 61]))

Meoww


ENCODING ENTIRE TEXT DATASET and then store it into torch.Tensor

In [208]:
data = torch.tensor(encode(text), dtype =torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [209]:
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [210]:
# train, val = train_test_split(data, train_size=0.9, random_state=0)

n = int(0.9*len(data)) # first 90% will be train
train = data[:n]
val= data[n:]

In [211]:
print(train.shape)
print(val.shape)

torch.Size([1003854])
torch.Size([111540])


In [212]:
# block_size = 8
# train[:block_size+1] # efficiency + for making TF see context from as little as 1 to block size

#tf will never see more than block_size input and op is truncated. context for any thing between 1 to blocksize can be inferenced

we will have mini batches of multiple chunks of text stacked up on a single tensor. FOR efficiency and parallelization. KEEP GPU BUSY.

In [213]:
def get_batch (split):
    data = train if split == 'train' else val
    ix = torch.randint( len(data) - block_size, (batch_size,))
    x = torch.stack(
        [data[i: i+block_size] for i in ix]
    )
    y = torch.stack(
            [data[i+1: i+block_size+1] for i in ix]
    )
    x, y = x.to(device), y.to(device)
    return x, y

In [214]:
@torch.no_grad() #dont call backward which is mroe efficet, no need to store intermediate var
def estimate_loss():
    output = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[i] = loss.item()
        output[split] = losses.mean()
    model.train()
    return output

In [215]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        #idx and targets are both (B, T) tensor
        logits = self.token_embedding_table(idx)  #(B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [216]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

In [217]:
optimizer = torch.optim.AdamW(m.parameters(), lr = lr) #3e-4 for most networks but for small this works or even higher

In [218]:
for iter in range(max_iters):

    #every once in a while eval the loss on train and val sets
    if iter %eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    #sample a batch of data
    xb, yb = get_batch('train')

    #eval the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) #setting grads in previous step to zero
    loss.backward() #get grad for all the param
    optimizer.step() #use grad to update param



step 0: train loss 4.6463, val loss 4.6247
step 300: train loss 2.8133, val loss 2.8240
step 600: train loss 2.5560, val loss 2.5559
step 900: train loss 2.5022, val loss 2.5201
step 1200: train loss 2.4703, val loss 2.4986
step 1500: train loss 2.4757, val loss 2.5012
step 1800: train loss 2.4784, val loss 2.4908
step 2100: train loss 2.4723, val loss 2.4953
step 2400: train loss 2.4599, val loss 2.4889
step 2700: train loss 2.4636, val loss 2.4835


In [219]:
context = torch.zeros((1, 1), dtype=torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=300)[0].tolist()))



Therourf.
BENRO, ive roanothallauo-d I th ttorat il me heson, d
Freu whay
US:XEORICHanowe par d ncay be m ar gr s ceanenegato! ofarne rermored; iny'n D wathece me, tnvyo an w,
Tea, lodirsclleendsut arshawee s mak bowes o'sand ms lat,
em t m tex?
O: ow an he, arvets thasoonthe mesthef fourme?
Ifar o
